# Text Classification

**Text classification** assigns predefined categories to documents, sentences, or messages based on their content. It is one of the most widely deployed NLP tasks — spam filters, sentiment analyzers, topic taggers, and intent detectors all reduce to classification.

Given a piece of text and a set of possible labels, the model learns which label best describes the input. The same framework handles binary decisions (*spam or not spam*), multi-class categorization (*sports, politics, or technology*), and multi-label assignment (*a document tagged with both "finance" and "technology"*).

This notebook covers the full text classification workflow: preparing data, building baselines with TF-IDF and classical classifiers, evaluating with the right metrics, handling class imbalance, and analyzing errors to improve performance.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Types of Text Classification

| Type | Labels per Document | Example |
|------|-------------------|--------|
| **Binary** | Exactly one of two | Spam vs not spam |
| **Multi-class** | Exactly one of $K$ classes | News topic: sports, politics, tech |
| **Multi-label** | Zero or more of $K$ labels | Tags: "AI", "Python", "tutorial" |

Binary classification is the simplest case. Multi-class extends this to $K > 2$ mutually exclusive categories. Multi-label allows a document to belong to multiple categories simultaneously — common in tagging systems and content moderation where a post may violate several policies at once.

Most classical algorithms handle multi-class natively (softmax, one-vs-rest). Multi-label typically trains one binary classifier per label.

---

## 2. Common Applications

**Sentiment analysis** — Determine whether text expresses positive, negative, or neutral opinion. Used for product reviews, social media monitoring, and brand tracking.

**Spam and abuse detection** — Filter unwanted email, comments, or messages. Often binary, with severe class imbalance (most messages are legitimate).

**Topic classification** — Assign documents to subject categories. Powers news aggregators, content recommendation, and document routing.

**Intent detection** — Classify user utterances in chatbots (*"book a flight"*, *"check balance"*, *"talk to agent"*).

**Language identification** — Detect which language a text is written in.

**Emotion detection** — Finer-grained than sentiment: joy, anger, fear, sadness.

Despite different domains, the underlying pipeline is the same: represent text numerically, train a classifier, evaluate on held-out data.

---

## 3. The Classification Pipeline

```
Raw text + labels
       ↓
Train / validation / test split
       ↓
Text vectorization (TF-IDF, embeddings)
       ↓
Classifier training (logistic regression, naive Bayes, SVM)
       ↓
Evaluation on validation / test set
       ↓
Error analysis → iterate
```

scikit-learn's `Pipeline` chains vectorization and classification into a single object — ensuring that test data is transformed with statistics learned only from training data, and simplifying deployment.

---

## 4. A Sentiment Classification Dataset

We start with a binary sentiment task — classifying movie reviews as positive or negative.

In [ ]:
reviews = [
    ("this movie is fantastic and absolutely amazing", "positive"),
    ("wonderful acting with a great storyline throughout", "positive"),
    ("best film i have ever seen in my life", "positive"),
    ("brilliant performance by the entire cast", "positive"),
    ("absolutely loved every single minute of it", "positive"),
    ("outstanding masterpiece of modern cinema", "positive"),
    ("excellent cinematography and beautiful direction", "positive"),
    ("really enjoyed this film thoroughly", "positive"),
    ("a delightful and heartwarming experience", "positive"),
    ("superb writing with compelling characters", "positive"),
    ("terrible movie and a complete waste of time", "negative"),
    ("awful acting with a boring and dull plot", "negative"),
    ("worst film ever made do not watch", "negative"),
    ("horrible experience i want my money back", "negative"),
    ("completely disappointed and very frustrated", "negative"),
    ("dreadful and painful to sit through", "negative"),
    ("poor script with weak and flat characters", "negative"),
    ("boring predictable and utterly pointless", "negative"),
    ("a total disaster from start to finish", "negative"),
    ("unwatchable garbage with no redeeming qualities", "negative"),
    ("decent film with some good moments overall", "positive"),
    ("pretty good and enjoyable watch overall", "positive"),
    ("not great but not terrible either honestly", "negative"),
    ("mediocre at best with nothing special", "negative"),
    ("mixed feelings but leaning slightly positive", "positive"),
    ("could have been better but still okay", "negative"),
    ("solid effort with a few standout scenes", "positive"),
    ("forgettable and largely uninspired work", "negative"),
    ("genuinely moving and well crafted story", "positive"),
    ("lazy writing ruins an otherwise fine concept", "negative"),
]

df = pd.DataFrame(reviews, columns=["text", "label"])
print(f"Dataset: {len(df)} reviews")
print(df["label"].value_counts())
print(f"\nSample positive: {df[df.label=='positive'].text.iloc[0]}")
print(f"Sample negative: {df[df.label=='negative'].text.iloc[0]}")

---

## 5. Train-Test Split

Split data before any preprocessing that learns from the corpus. Use **stratified splitting** to preserve class proportions in each split — especially important with imbalanced data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"],
    test_size=0.25, random_state=42, stratify=df["label"]
)

print(f"Training: {len(X_train)} samples")
print(f"Test:     {len(X_test)} samples")
print(f"\nTrain class distribution:\n{pd.Series(y_train).value_counts()}")

---

## 6. Baseline: TF-IDF + Logistic Regression

**Logistic regression** on TF-IDF features is the standard text classification baseline. It is fast, interpretable, and often hard to beat on small to medium datasets.

The model learns a weight for each word (or n-gram). Positive weights push toward the positive class; negative weights push toward the negative class.

In [ ]:
pipe_lr = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

pipe_lr.fit(X_train, y_train)
y_pred = pipe_lr.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"\n{classification_report(y_test, y_pred)}")

---

## 7. Comparing Classifiers

Three classical classifiers dominate text classification:

| Classifier | Strengths | Weaknesses |
|-----------|------------|------------|
| **Logistic Regression** | Fast, probabilistic output, interpretable | Linear decision boundary |
| **Multinomial Naive Bayes** | Very fast, works with small data | Independence assumption |
| **Linear SVM** | Strong with high-dimensional sparse data | No native probability scores |

In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42, max_iter=2000),
}

print(f"{'Classifier':<22s} {'Accuracy':>8s} {'F1':>8s}")
print("-" * 40)
for name, clf in classifiers.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
        ("clf", clf),
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    print(f"{name:<22s} {acc:8.3f} {f1:8.3f}")

---

## 8. Evaluation Metrics

Accuracy alone is misleading when classes are imbalanced. Use metrics matched to your task:

**Precision** — Of predicted positives, how many are correct?

$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall** — Of actual positives, how many did we find?

$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1-score** — Harmonic mean of precision and recall.

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**When to prioritize what:**
- **High precision** — spam filter (avoid false positives blocking legitimate mail).
- **High recall** — disease screening (catch every case, even at cost of false alarms).
- **F1** — balanced need when classes are roughly equal.

In [ ]:
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

cm = confusion_matrix(y_test, y_pred, labels=["negative", "positive"])
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["negative", "positive"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — Sentiment Classification")
plt.tight_layout()
plt.show()

---

## 9. Interpreting Feature Weights

A major advantage of linear classifiers on TF-IDF features: **interpretability**. The learned coefficients reveal which words drive each class prediction.

In [ ]:
tfidf = pipe_lr.named_steps["tfidf"]
clf = pipe_lr.named_steps["clf"]
features = tfidf.get_feature_names_out()
coefs = clf.coef_[0]

top_pos = np.argsort(coefs)[-10:][::-1]
top_neg = np.argsort(coefs)[:10]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].barh([features[i] for i in top_pos[::-1]], coefs[top_pos[::-1]], color="green", alpha=0.7)
axes[0].set_title("Top positive indicators")
axes[0].set_xlabel("Coefficient")

axes[1].barh([features[i] for i in top_neg], coefs[top_neg], color="red", alpha=0.7)
axes[1].set_title("Top negative indicators")
axes[1].set_xlabel("Coefficient")

plt.suptitle("What the classifier learned", y=1.02)
plt.tight_layout()
plt.show()

---

## 10. The Effect of N-grams

Unigrams alone miss phrases like *"not good"* and *"waste of time"*. Adding bigrams captures negation and multi-word expressions.

In [ ]:
ngram_configs = [
    ("Unigrams only", (1, 1)),
    ("Unigrams + bigrams", (1, 2)),
    ("Bigrams only", (2, 2)),
]

print(f"{'N-gram config':<22s} {'Features':>8s} {'Accuracy':>8s} {'F1':>8s}")
print("-" * 50)
for name, ngram_range in ngram_configs:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=ngram_range)),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    n_features = len(pipe.named_steps["tfidf"].get_feature_names_out())
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    print(f"{name:<22s} {n_features:8d} {acc:8.3f} {f1:8.3f}")

---

## 11. Cross-Validation

A single train-test split can be lucky or unlucky. **Cross-validation** trains and evaluates on multiple splits, giving a more reliable performance estimate.

**Stratified k-fold** preserves class proportions in each fold — essential for imbalanced datasets.

In [ ]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, df["text"], df["label"], cv=cv, scoring="f1_weighted")

print(f"5-fold CV F1 scores: {scores}")
print(f"Mean F1: {scores.mean():.3f} (+/- {scores.std():.3f})")

---

## 12. Multi-Class Classification

Extend to $K$ mutually exclusive classes. Logistic regression uses **softmax** to produce a probability distribution over all classes. One-vs-rest (OvR) trains $K$ binary classifiers — one per class.

In [ ]:
news = [
    ("the team scored a winning goal in the final minute", "sports"),
    ("championship game ended with a dramatic overtime victory", "sports"),
    ("star player signed a record breaking contract today", "sports"),
    ("the tournament draw was announced for next month", "sports"),
    ("fans celebrated the historic championship win downtown", "sports"),
    ("the senate passed a new healthcare reform bill", "politics"),
    ("election polls show a tight race between candidates", "politics"),
    ("the president announced new foreign policy measures", "politics"),
    ("parliament debated the proposed tax legislation today", "politics"),
    ("voters head to polls in the critical midterm election", "politics"),
    ("the company launched a revolutionary new smartphone", "tech"),
    ("artificial intelligence startup raised fifty million dollars", "tech"),
    ("new chip architecture promises faster computing speeds", "tech"),
    ("the software update fixes critical security vulnerabilities", "tech"),
    ("tech giants compete for dominance in cloud computing", "tech"),
    ("scientists discovered a new species in the rainforest", "science"),
    ("the space telescope captured images of distant galaxies", "science"),
    ("researchers published findings on climate change impact", "science"),
    ("a breakthrough in quantum computing was announced today", "science"),
    ("the medical trial showed promising results for patients", "science"),
]

news_df = pd.DataFrame(news, columns=["text", "topic"])
X_tr, X_te, y_tr, y_te = train_test_split(
    news_df["text"], news_df["topic"],
    test_size=0.25, random_state=42, stratify=news_df["topic"]
)

pipe_mc = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
pipe_mc.fit(X_tr, y_tr)
y_pred_mc = pipe_mc.predict(X_te)

print(classification_report(y_te, y_pred_mc))

cm_mc = confusion_matrix(y_te, y_pred_mc, labels=sorted(news_df["topic"].unique()))
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm_mc, display_labels=sorted(news_df["topic"].unique()))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Multi-class Confusion Matrix")
plt.tight_layout()
plt.show()

---

## 13. Class Imbalance

Real-world datasets are often imbalanced — spam filters see 99% legitimate mail; fraud detection sees 0.1% fraud. A classifier predicting the majority class always achieves high accuracy but fails completely on the minority class.

**Strategies:**
- Use **F1, precision, recall** instead of accuracy.
- **Class weights** — penalize misclassification of minority class more heavily.
- **Oversampling** minority class or **undersampling** majority class.
- **SMOTE** — synthetic minority oversampling.
- Adjust **decision threshold** based on precision-recall tradeoff.

In [ ]:
# Imbalanced spam dataset: 90% ham, 10% spam
spam_data = (
    [("hey are we still meeting for lunch tomorrow", "ham")] * 18
    + [("please review the attached report by friday", "ham")] * 18
    + [("meeting rescheduled to three pm in conference room", "ham")] * 18
    + [("thanks for sending the documents over", "ham")] * 18
    + [("win free money click here now limited offer", "spam")] * 6
    + [("congratulations you won a prize claim immediately", "spam")] * 6
    + [("buy cheap pills online no prescription needed", "spam")] * 6
)

spam_df = pd.DataFrame(spam_data, columns=["text", "label"])
print(f"Class distribution:\n{spam_df['label'].value_counts()}\n")

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    spam_df["text"], spam_df["label"], test_size=0.25, random_state=42, stratify=spam_df["label"]
)

for weighted in [False, True]:
    clf = LogisticRegression(
        max_iter=1000, random_state=42,
        class_weight="balanced" if weighted else None
    )
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english")),
        ("clf", clf),
    ])
    pipe.fit(X_tr_s, y_tr_s)
    preds = pipe.predict(X_te_s)
    label = "With class_weight='balanced'" if weighted else "No class weighting"
    print(f"{label}:")
    print(classification_report(y_te_s, preds))
    print()

---

## 14. Probability Thresholds

Logistic regression outputs class probabilities. The default threshold is 0.5 — but you can adjust it to favor precision or recall.

In [ ]:
pipe_prob = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
pipe_prob.fit(X_train, y_train)

probs = pipe_prob.predict_proba(X_test)
classes = pipe_prob.classes_
pos_idx = list(classes).index("positive")
pos_probs = probs[:, pos_idx]

print(f"{'Threshold':>10s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s}")
print("-" * 44)
for threshold in [0.3, 0.5, 0.7, 0.9]:
    preds_t = np.where(pos_probs >= threshold, "positive", "negative")
    prec = precision_score(y_test, preds_t, pos_label="positive", zero_division=0)
    rec = recall_score(y_test, preds_t, pos_label="positive", zero_division=0)
    f1 = f1_score(y_test, preds_t, pos_label="positive", zero_division=0)
    print(f"{threshold:10.1f} {prec:10.3f} {rec:10.3f} {f1:10.3f}")

---

## 15. Error Analysis

Examining misclassified examples reveals systematic failures — negation, sarcasm, domain-specific language, or ambiguous labels. Error analysis drives targeted improvements: better preprocessing, more training data for failure modes, or feature engineering.

In [ ]:
errors = []
for text, true, pred in zip(X_test, y_test, y_pred):
    if true != pred:
        prob = pipe_lr.predict_proba([text])[0]
        conf = prob.max()
        errors.append({"text": text, "true": true, "predicted": pred, "confidence": conf})

if errors:
    err_df = pd.DataFrame(errors).sort_values("confidence", ascending=False)
    print(f"Misclassified examples ({len(errors)} total):\n")
    for _, row in err_df.iterrows():
        print(f"  Text:       {row['text']}")
        print(f"  True:       {row['true']}  →  Predicted: {row['predicted']}  (conf: {row['confidence']:.2f})")
        print()
else:
    print("No misclassifications on the test set!")
    print("\nAmbiguous examples that are inherently hard to classify:")
    hard = [
        "not great but not terrible either honestly",
        "mixed feelings but leaning slightly positive",
    ]
    for text in hard:
        pred = pipe_lr.predict([text])[0]
        prob = pipe_lr.predict_proba([text])[0]
        print(f"  '{text}'")
        print(f"    → {pred} (confidence: {prob.max():.2f})")

---

## 16. Multi-Label Classification

In multi-label classification, each document can have multiple labels. Train one binary classifier per label using `MultiLabelBinarizer` and `OneVsRestClassifier`.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier

tagged = [
    ("introduction to python programming for beginners", ["python", "tutorial"]),
    ("advanced machine learning with scikit learn", ["python", "ml"]),
    ("deep learning neural networks explained simply", ["ml", "tutorial"]),
    ("building a web app with flask and python", ["python", "web"]),
    ("natural language processing tutorial for beginners", ["ml", "tutorial", "nlp"]),
    ("deploying ml models to production environments", ["ml", "web"]),
    ("python data analysis with pandas library", ["python", "tutorial"]),
    ("sentiment analysis using machine learning techniques", ["ml", "nlp"]),
]

tag_texts = [t for t, _ in tagged]
tag_labels = [tags for _, tags in tagged]

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(tag_labels)
print(f"Labels: {list(mlb.classes_)}")
print(f"Label matrix shape: {Y.shape}\n")

pipe_ml = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))),
])
pipe_ml.fit(tag_texts, Y)

test_docs = [
    "learn python programming step by step",
    "neural network model for text classification",
]

for doc in test_docs:
    pred = pipe_ml.predict([doc])[0]
    tags = [mlb.classes_[i] for i, v in enumerate(pred) if v == 1]
    print(f"'{doc}'")
    print(f"  → tags: {tags}\n")

---

## 17. Practical Workflow

1. **Define the task** — binary, multi-class, or multi-label? What metric matters?
2. **Explore the data** — class distribution, text length, language, noise.
3. **Establish a baseline** — TF-IDF + logistic regression with cross-validation.
4. **Evaluate properly** — stratified splits, task-appropriate metrics, confusion matrix.
5. **Analyze errors** — inspect misclassifications for patterns.
6. **Iterate** — try n-grams, class weights, more data, or better preprocessing.
7. **Compare against baseline** — only deploy improvements that beat the baseline consistently.
8. **Test on held-out data** — evaluate once on a test set never used during development.

---

## 18. Beyond Classical Methods

Classical TF-IDF + linear classifiers remain strong baselines. When more performance is needed:

- **Fine-tuned language models** (BERT, DistilBERT) — contextual representations, state-of-the-art on most benchmarks.
- **Few-shot classification with LLMs** — classify from instructions and examples without training.
- **Ensemble methods** — combine multiple classifiers for robustness.

The classical pipeline is faster, cheaper, more interpretable, and often sufficient — especially with limited data and compute. Start simple; add complexity only when the baseline is clearly insufficient.

---

## 19. Summary

**Text classification** assigns predefined labels to text — powering sentiment analysis, spam detection, topic tagging, and intent recognition. Tasks may be binary, multi-class, or multi-label.

The standard pipeline vectorizes text with **TF-IDF**, trains a **linear classifier** (logistic regression, naive Bayes, or SVM), and evaluates with **precision, recall, and F1** — not accuracy alone. **N-grams**, **class weights**, and **threshold tuning** address common challenges.

**Cross-validation** gives reliable performance estimates. **Error analysis** on misclassified examples drives improvement. **Multi-label** tasks use one-vs-rest binary classifiers per label.

Classical methods are fast, interpretable, and effective. They establish the baseline that more complex approaches must beat to justify their additional cost.